# VocalMind GPU server

1. Settings: **Accelerator = GPU (T4/P100)**, **Internet = on**.
2. Add-ons → Secrets: attach `HF_TOKEN`, `NGROK_TOKEN`, `NGROK_DOMAIN`, `BACKEND_URL` (backend public URL + `/api/v1/internal/set-kaggle-url`).
3. Run all. The last cell blocks while the server is up; interrupt it to stop.

No kernel restart is needed: numpy/torch are pinned to Kaggle's preinstalled versions and the server runs in a fresh `python` process.

In [ ]:
BRANCH = "main"
!rm -rf VocalMind && git clone --depth 1 -b {BRANCH} https://github.com/galalqassas/VocalMind.git
!pip freeze | grep -iE '^(numpy|torch|torchaudio|torchvision)==' > constraints.txt
!pip install -q -r VocalMind/services/kaggle/requirements.txt -c constraints.txt

In [ ]:
import os
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
for name in ("HF_TOKEN", "NGROK_TOKEN", "NGROK_DOMAIN"):
    os.environ[name] = secrets.get_secret(name)
try:  # optional: lets the server register itself with a public backend
    os.environ["BACKEND_URL"] = secrets.get_secret("BACKEND_URL")
except Exception:
    print("No BACKEND_URL secret: set KAGGLE_SERVER_URL on the backend yourself.")

!python VocalMind/services/kaggle/server.py